# 실습 4: 영수증 / 인보이스 분석 (Expense Analysis)
**소요시간: 40분** | 난이도: ⭐⭐⭐

## 학습 목표
1. `analyze_expense` API로 영수증과 인보이스에서 구조화된 데이터를 추출합니다.
2. SummaryFields(요약 항목)와 LineItemGroups(품목 목록)를 파싱합니다.
3. 추출된 데이터를 pandas DataFrame으로 정리합니다.

## API 개요: analyze_expense
```python
response = textract.analyze_expense(
    Document={'Bytes': <바이트>}
)
```

### 응답 구조
```
response
└── ExpenseDocuments[]
    ├── SummaryFields[]          ← 헤더 정보 (벤더명, 날짜, 총액 등)
    │   ├── Type.Text            ← 필드 유형 (VENDOR_NAME, TOTAL, DATE 등)
    │   ├── ValueDetection.Text  ← 추출된 값
    │   └── Confidence           ← 신뢰도
    └── LineItemGroups[]         ← 품목 그룹
        └── LineItems[]          ← 개별 품목
            └── LineItemExpenseFields[]
                ├── Type.Text    ← ITEM, QUANTITY, PRICE, UNIT_PRICE 등
                └── ValueDetection.Text
```

### 주요 SummaryField 타입
| 타입 | 설명 |
|---|---|
| `VENDOR_NAME` | 판매처/벤더명 |
| `INVOICE_RECEIPT_DATE` | 날짜 |
| `INVOICE_RECEIPT_ID` | 영수증/인보이스 번호 |
| `TOTAL` | 총 금액 |
| `SUBTOTAL` | 소계 |
| `TAX` | 세금 |
| `AMOUNT_PAID` | 결제 금액 |


In [ ]:
# ✅ [제공 코드]
import boto3, os, json
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd

textract = boto3.client('textract', region_name='ap-northeast-2')

IMAGE_DIR = './images/'
image_path = os.path.join(IMAGE_DIR, 'lab04_receipt.jpg')

def load_document_bytes(path):
    with open(path, 'rb') as f:
        return f.read()

doc_bytes = load_document_bytes(image_path)

img = Image.open(image_path)
plt.figure(figsize=(6, 10))
plt.imshow(img)
plt.axis('off')
plt.title('원본 영수증/인보이스')
plt.show()


## ✏️ TODO 1: analyze_expense API 호출


In [ ]:
# ✏️ TODO 1: API를 호출하고 ExpenseDocument 수를 확인하세요
response = textract._____(                      # ← API 메서드명 (analyze_expense)
    Document={'Bytes': _____}                   # ← doc_bytes
)

expense_docs = response[_____]                  # ← 'ExpenseDocuments'
print(f"감지된 문서 수: {len(expense_docs)}개")

# 첫 번째 문서 사용
doc = expense_docs[0]
print(f"SummaryFields 수: {len(doc.get('SummaryFields', []))}개")
print(f"LineItemGroups 수: {len(doc.get('LineItemGroups', []))}개")


## ✏️ TODO 2: SummaryFields — 헤더 정보 추출

영수증의 주요 정보(벤더명, 날짜, 총액 등)를 추출하세요.


In [ ]:
# ✏️ TODO 2: SummaryFields에서 주요 정보를 추출하세요
summary_fields = doc.get('SummaryFields', [])
summary = {}

for field in summary_fields:
    field_type = field.get('Type', {}).get(_____, 'UNKNOWN')       # ← 'Text'
    field_value = field.get('ValueDetection', {}).get(_____, '')   # ← 'Text'
    field_conf = field.get('ValueDetection', {}).get(_____, 0)     # ← 'Confidence'
    
    summary[field_type] = {
        'value': field_value,
        'confidence': field_conf
    }

# 주요 필드 출력
key_fields = ['VENDOR_NAME', 'INVOICE_RECEIPT_DATE', 'INVOICE_RECEIPT_ID',
              'TOTAL', 'SUBTOTAL', 'TAX', 'AMOUNT_PAID']

print("영수증 요약 정보:")
print("-" * 50)
for key in key_fields:
    if key in summary:
        val = summary[key]['value']
        conf = summary[key]['confidence']
        print(f"  {key:<30}: {val} ({conf:.1f}%)")

print(f"\n전체 감지된 필드 수: {len(summary)}개")
print(f"전체 필드 목록: {list(summary.keys())}")


## ✏️ TODO 3: LineItemGroups — 품목 목록 추출

각 품목(ITEM, QUANTITY, UNIT_PRICE, PRICE)을 추출하여 DataFrame으로 만드세요.


In [ ]:
# ✏️ TODO 3: 품목 목록을 추출하여 DataFrame을 만드세요
line_items = []

for group in doc.get(_____,  []):        # ← 'LineItemGroups'
    for item in group.get(_____,  []):    # ← 'LineItems'
        item_data = {}
        
        for field in item.get(_____,  []): # ← 'LineItemExpenseFields'
            field_type  = field.get('Type', {}).get(_____, '')        # ← 'Text'
            field_value = field.get('ValueDetection', {}).get(_____, '') # ← 'Text'
            item_data[field_type] = field_value
        
        if item_data:
            line_items.append(item_data)

print(f"감지된 품목 수: {len(line_items)}개")

if line_items:
    df_items = pd.DataFrame(line_items)
    print("\n품목 목록:")
    print(df_items.to_string(index=False))
else:
    print("품목 정보가 없습니다.")


## ✏️ TODO 4: 영수증 데이터 검증 — 합계 검증

품목별 PRICE의 합계가 SUBTOTAL과 일치하는지 검증하세요.


In [ ]:
# ✏️ TODO 4: PRICE 합계와 SUBTOTAL을 비교하여 검증하세요
import re

def parse_amount(text):
    """'$1,234.56' → 1234.56 변환"""
    if not text:
        return None
    cleaned = re.sub(r'[^\d.]', '', text)
    try:
        return float(cleaned)
    except:
        return None

# 품목 가격 합계 계산
total_from_items = 0
if line_items and 'PRICE' in pd.DataFrame(line_items).columns:
    for item in line_items:
        price = parse_amount(item.get(_____, ''))   # ← 'PRICE'
        if price:
            total_from_items += price

# SUBTOTAL과 TOTAL 가져오기
subtotal_text = summary.get('SUBTOTAL', {}).get(_____, '')   # ← 'value'
total_text    = summary.get('TOTAL',    {}).get(_____, '')   # ← 'value'

subtotal = parse_amount(subtotal_text)
total    = parse_amount(total_text)

print("💰 금액 검증:")
print("-" * 40)
print(f"  품목 합계: {total_from_items:.2f}")
print(f"  소계(SUBTOTAL): {subtotal}")
print(f"  총계(TOTAL): {total}")

# 검증
if subtotal and abs(total_from_items - subtotal) < 0.01:
    print("\n  ✅ 품목 합계 = 소계 (검증 통과)")
elif total_from_items > 0 and subtotal:
    diff = total_from_items - subtotal
    print(f"\n  ⚠️  차이 발생: {diff:.2f} (품목 누락 또는 OCR 오류 가능성)")
else:
    print("\n  ℹ️  비교할 품목 데이터 부족")


## ✏️ TODO 5: 결과 JSON 저장

영수증 분석 결과를 구조화된 JSON으로 저장하세요.


In [ ]:
# ✏️ TODO 5: 분석 결과를 JSON 파일로 저장하세요
result = {
    'summary': {k: v['value'] for k, v in _____.items()},   # ← summary
    'line_items': _____,                                     # ← line_items
    'validation': {
        'items_total': total_from_items,
        'subtotal': subtotal,
        'total': total
    }
}

with open('expense_result.json', 'w', encoding='utf-8') as f:
    json.dump(_____, f, ensure_ascii=False, indent=2)   # ← result

print("✅ expense_result.json 저장 완료")
print(json.dumps(result['summary'], ensure_ascii=False, indent=2))


## 💡 심화 도전
1. 여러 영수증을 일괄 처리하여 월별 지출 합계를 집계해보세요.
2. VENDOR_NAME 기준으로 영수증을 분류하는 코드를 작성해보세요.
3. 세금(TAX) 비율을 계산하여 예상 세율과 비교해보세요.
